# Data Augmentation para Landmarks de Mãos/Braços (LIBRAS)

Este notebook aplica estratégias de data augmentation em arquivos de landmarks gerados pelo pipeline do notebook de coleta (`create_landmarks.ipynb`).

- Compatível com o padrão de arquivos `.npy` (dict com 'hands' e opcionalmente 'arms', ou apenas array de mãos).
- Estratégias aplicadas:
  - Ruído Gaussiano controlado
  - Escala leve
  - Rotação suave (eixo Z)
  - Time warping (para sequências)

In [ ]:
import os
import numpy as np
import shutil
import pandas as pd

# --- CAMINHOS ---
AUGMENTED_DIR = '../dataset/processed/landmarks_augmented'
LABELS_CSV_IN = '../landmark_labels_vlisbrasil.csv'
AUGMENTED_LABELS_CSV_OUT = os.path.join(os.path.dirname(AUGMENTED_DIR), 'augmented_landmark_labels.csv')

os.makedirs(AUGMENTED_DIR, exist_ok=True)

# --- FUNÇÕES DE AUMENTO DE DADOS ---

def add_gaussian_noise(landmarks, std=0.005):
    noise = np.random.normal(0, std, landmarks.shape)
    return landmarks + noise

def scale_landmarks(landmarks, scale_range=(0.95, 1.05)):
    center = landmarks.mean(axis=0, keepdims=True)
    scale = np.random.uniform(*scale_range)
    return (landmarks - center) * scale + center

def rotate_landmarks_z(landmarks, angle_range=(-10, 10)):
    theta = np.radians(np.random.uniform(*angle_range))
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    
    center = landmarks[:, :2].mean(axis=0, keepdims=True)
    xy = landmarks[:, :2] - center
    xy_rot = xy @ R.T + center

    landmarks_aug = landmarks.copy()
    landmarks_aug[:, :2] = xy_rot
    return landmarks_aug

def mirror_landmarks_horizontally(landmarks_sequence):
    mirrored_sequence = landmarks_sequence.copy()
    mirrored_sequence[:, :, 0] = 1 - mirrored_sequence[:, :, 0]
    return mirrored_sequence

def augment_frame(frame_landmarks):
    lm = frame_landmarks.copy()
    lm = add_gaussian_noise(lm)
    lm = scale_landmarks(lm)
    lm = rotate_landmarks_z(lm)
    return lm

# --- FUNÇÃO PRINCIPAL DE AUMENTO ---

def process_and_augment_landmarks(landmarks_data, base_name, save_dir, n_aug=3):
    saved_files = []
    for i in range(n_aug):
        augmented_sequence = np.array([augment_frame(frame) for frame in landmarks_data])
        save_path = os.path.join(save_dir, f'{base_name}_aug{i+1}.npy')
        np.save(save_path, augmented_sequence)
        saved_files.append(save_path)
    return saved_files

# --- EXECUÇÃO ---

# Carrega o arquivo de labels
try:
    labels_df = pd.read_csv(LABELS_CSV_IN)
except FileNotFoundError:
    print(f"Erro: Arquivo de labels não encontrado em '{LABELS_CSV_IN}'")
    # Interrompe a execução se o arquivo não existir
    labels_df = pd.DataFrame()

if not labels_df.empty:
    print(f'Encontrados {len(labels_df)} registros no CSV para aumentar.')

    # Limpa o diretório de augmentados antes de começar
    if os.path.exists(AUGMENTED_DIR):
        shutil.rmtree(AUGMENTED_DIR)
    os.makedirs(AUGMENTED_DIR)

    new_labels_list = []

    for index, row in labels_df.iterrows():
        original_landmark_path = row['landmark_path']
        
        if not os.path.exists(original_landmark_path):
            print(f"Aviso: Arquivo de landmark não encontrado: {original_landmark_path}. Pulando.")
            continue

        base_name = os.path.splitext(os.path.basename(original_landmark_path))[0]
        landmarks_data = np.load(original_landmark_path)

        # 1. Aumentar a versão original
        augmented_files = process_and_augment_landmarks(landmarks_data, base_name, AUGMENTED_DIR, n_aug=3)
        for new_path in augmented_files:
            new_labels_list.append({
                'video_path': row['video_path'],
                'landmark_path': new_path,
                'file_id': row['file_id'],
                'participant': row['participant'],
                'sign_id': row['sign_id'],
                'phrase': row['phrase']
            })

        # 2. Aumentar a versão espelhada
        mirrored_landmarks = mirror_landmarks_horizontally(landmarks_data)
        mirrored_base_name = f"{base_name}_mirror"
        mirrored_augmented_files = process_and_augment_landmarks(mirrored_landmarks, mirrored_base_name, AUGMENTED_DIR, n_aug=3)
        for new_path in mirrored_augmented_files:
            new_labels_list.append({
                'video_path': row['video_path'],
                'landmark_path': new_path,
                'file_id': row['file_id'],
                'participant': row['participant'],
                'sign_id': row['sign_id'],
                'phrase': row['phrase']
            })

    # Cria e salva o novo DataFrame de labels
    augmented_labels_df = pd.DataFrame(new_labels_list)
    augmented_labels_df.to_csv(AUGMENTED_LABELS_CSV_OUT, index=False)

    print(f'\nAumento de dados finalizado.')
    print(f'{len(augmented_labels_df)} arquivos aumentados foram gerados em: {AUGMENTED_DIR}')
    print(f'Novo arquivo de labels salvo em: {AUGMENTED_LABELS_CSV_OUT}')